# Adapter Evaluation

This notebook reads the LoRA adapter zip and `golden_prompts.jsonl` from Google Drive, loads the base model with the adapter, and runs the same rule-based checks used by the local app eval.

In [ ]:
!pip install -U --force-reinstall "sympy>=1.13.3,<1.14"
!pip install -U "transformers>=4.46" "accelerate>=1" "peft>=0.13" "bitsandbytes>=0.44" safetensors

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path('/content/drive/MyDrive/colab Notebooks/codex-artifacts/adapters/qwen25-coder-3b-sql-python-lora.zip')
eval_path = Path('/content/drive/MyDrive/colab Notebooks/codex-artifacts/evals/golden_prompts.jsonl')

assert zip_path.exists(), f'Missing adapter zip: {zip_path}'
assert eval_path.exists(), f'Missing eval file: {eval_path}'

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall('adapter_artifacts')

adapter_dir = next(Path('adapter_artifacts').glob('qwen25-coder-3b-sql-python-lora'))
assert (adapter_dir / 'adapter_model.safetensors').exists(), 'Missing adapter_model.safetensors'
assert (adapter_dir / 'adapter_config.json').exists(), 'Missing adapter_config.json'

print('adapter zip:', zip_path)
print('evals:', eval_path)
print('adapter:', adapter_dir)
print('evals:', eval_path)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model = 'Qwen/Qwen2.5-Coder-3B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
print('loaded adapter')

In [ ]:
import json

SYSTEM = 'You are a direct SQL/Python data engineering assistant. Return concise, correct answers. Do not invent APIs.'

def generate(prompt, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def check(case, answer):
    normalized = answer.lower()
    missing = [item for item in case.get('required', []) if item.lower() not in normalized]
    forbidden = [item for item in case.get('forbidden', []) if item.lower() in normalized]
    return not missing and not forbidden, missing, forbidden

cases = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines() if line.strip()]
results = []
for case in cases:
    answer = generate(case['prompt'])
    passed, missing, forbidden = check(case, answer)
    results.append({
        'id': case['id'],
        'passed': passed,
        'missing_required': missing,
        'present_forbidden': forbidden,
        'answer': answer,
    })
    print(case['id'], 'PASS' if passed else 'FAIL')
    print(answer[:1000])
    print('-' * 80)

passed = sum(1 for result in results if result['passed'])
print(f'passed {passed}/{len(results)}')
Path('adapter_eval_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
files.download('adapter_eval_results.json')